In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.cluster import KMeans

# -----------------------------
# Graph Attention Layer (GAT)
# -----------------------------
class GraphAttentionLayer(nn.Module):
    def __init__(self, in_features, out_features, dropout, alpha, concat=True):
        super(GraphAttentionLayer, self).__init__()
        self.dropout = dropout
        self.alpha = alpha
        self.concat = concat

        self.W = nn.Parameter(torch.empty(size=(in_features, out_features)))
        nn.init.xavier_uniform_(self.W.data, gain=1.414)

        self.a = nn.Parameter(torch.empty(size=(2 * out_features, 1)))
        nn.init.xavier_uniform_(self.a.data, gain=1.414)

        self.leakyrelu = nn.LeakyReLU(self.alpha)

    def forward(self, h, adj):
        Wh = torch.mm(h, self.W)  # (N, out_features)
        a_input = self._prepare_attentional_mechanism_input(Wh)
        e = self.leakyrelu(torch.matmul(a_input, self.a).squeeze(2))
        
        # Use a large negative value for non-adjacent nodes
        zero_vec = -9e15 * torch.ones_like(e)
        attention = torch.where(adj > 0, e, zero_vec)
        attention = F.softmax(attention, dim=1)
        attention = F.dropout(attention, self.dropout, training=self.training)
        h_prime = torch.matmul(attention, Wh)
        return F.relu(h_prime) if self.concat else h_prime

    def _prepare_attentional_mechanism_input(self, Wh):
        N = Wh.size()[0]
        Wh_repeated_in_chunks = Wh.repeat_interleave(N, dim=0)
        Wh_repeated_alternating = Wh.repeat(N, 1)
        all_combinations_matrix = torch.cat([Wh_repeated_in_chunks, Wh_repeated_alternating], dim=1)
        return all_combinations_matrix.view(N, N, 2 * self.W.shape[1])

    def __repr__(self):
        return self.__class__.__name__ + ' (' + str(self.W.shape[0]) + ' -> ' + str(self.W.shape[1]) + ')'


# -----------------------------
# Inner Product Decoder
# -----------------------------
class InnerProductDecoder(nn.Module):
    def __init__(self, dropout, act=torch.sigmoid):
        super(InnerProductDecoder, self).__init__()
        self.dropout = dropout
        self.act = act

    def forward(self, z):
        z = F.dropout(z, self.dropout, training=self.training)
        adj = self.act(torch.mm(z, z.t()))
        return adj


# -----------------------------
# GraphGAT: GAT Module for One Omics
# -----------------------------
class GraphGAT(nn.Module):
    def __init__(self, nfeat, nhid, nclass, dropout, alpha, nheads, npatient):
        super(GraphGAT, self).__init__()
        self.dropout = dropout
        self.attentions = nn.ModuleList([
            GraphAttentionLayer(nfeat, nhid, dropout=dropout, alpha=alpha, concat=True)
            for _ in range(nheads)
        ])
        self.out_att = GraphAttentionLayer(nhid * nheads, nclass, dropout=dropout, alpha=alpha, concat=False)
        
        # Omics-level attention fusion parameters
        attention_size = 16
        self.Wz = nn.Parameter(torch.empty(size=(nclass, attention_size)))
        nn.init.xavier_uniform_(self.Wz.data, gain=1.414)
        self.Wa = nn.Parameter(torch.empty(size=(npatient, attention_size)))
        nn.init.xavier_uniform_(self.Wa.data, gain=1.414)
        self.v = nn.Parameter(torch.empty(size=(attention_size, 1)))
        nn.init.xavier_uniform_(self.v.data, gain=1.414)

    def forward(self, x, adj):
        z = F.dropout(x, self.dropout, training=self.training)
        z = torch.cat([att(z, adj) for att in self.attentions], dim=1)
        z = F.dropout(z, self.dropout, training=self.training)
        z = F.elu(self.out_att(z, adj))
        a = F.tanh(torch.mm(z, self.Wz) + torch.mm(adj, self.Wa))
        a = torch.mm(a, self.v)
        a = F.softmax(a, dim=1)
        a = a + a.T  # Enforce symmetry in attention
        z = torch.mm(a, z)
        return z


# -----------------------------
# MultiGATAE: The Full Model
# -----------------------------
class MultiGATAE(nn.Module):
    def __init__(self, nfeat, nhid, nclass, dropout, alpha, nheads, npatient):
        """
        Multi-omics Graph Attention Autoencoder.
        Expects three omics: mRNA, miRNA, and DNA methylation.
        """
        super(MultiGATAE, self).__init__()
        self.dropout = dropout
        
        self.RNASeq = GraphGAT(nfeat, nhid, nclass, dropout, alpha, nheads, npatient)
        self.MiRNA  = GraphGAT(nfeat, nhid, nclass, dropout, alpha, nheads, npatient)
        self.DNAm   = GraphGAT(nfeat, nhid, nclass, dropout, alpha, nheads, npatient)

        self.dc = InnerProductDecoder(dropout, act=lambda x: x)

    def forward(self, mrna, mirna, dnam, adj):
        x_r = self.RNASeq(mrna, adj)
        x_m = self.MiRNA(mirna, adj)
        x_d = self.DNAm(dnam, adj)
        # Fuse the three omics by averaging
        z = (x_r + x_m + x_d) / 3
        return self.dc(z), z


# -----------------------------
# Synthetic Data Generation
# -----------------------------
def generate_synthetic_omics(n_nodes, n_features, labels, noise=0.1):
    """
    Generate synthetic omics data with an underlying cluster structure.
    
    Args:
      n_nodes: Number of samples (patients).
      n_features: Feature dimension for the omics data.
      labels: Ground truth cluster labels (shared among omics).
      noise: Noise level.
      
    Returns:
      data: Tensor of shape (n_nodes, n_features)
    """
    # Define a low-dimensional latent space dimension
    d = 10  
    n_clusters = int(labels.max().item()) + 1
    # Generate cluster centers in the latent space
    centers = torch.randn(n_clusters, d)
    # For each node, use its cluster center with some noise
    latent = centers[labels] + noise * torch.randn(n_nodes, d)
    # Project latent representations to high-dimensional space
    proj = torch.randn(d, n_features)
    data = latent.mm(proj) + 0.1 * torch.randn(n_nodes, n_features)
    return data

def compute_adjacency_from_data(data, sigma=1.0):
    """
    Compute a similarity (adjacency) matrix from data using a Gaussian kernel.
    
    Args:
      data: Tensor of shape (n_nodes, n_features)
      sigma: Bandwidth parameter for the kernel.
      
    Returns:
      A: Symmetric similarity matrix of shape (n_nodes, n_nodes)
    """
    diff = data.unsqueeze(1) - data.unsqueeze(0)  # (n_nodes, n_nodes, n_features)
    dist_sq = torch.sum(diff ** 2, dim=2)
    A = torch.exp(-dist_sq / (2 * sigma ** 2))
    return A


# -----------------------------
# Main Training and Clustering Pipeline
# -----------------------------
# def main():
#     # Hyperparameters
#     n_nodes = 100        # Number of patients/samples
#     n_features = 500     # Feature dimension (for each omics)
#     nhid = 64            # Hidden units per attention head
#     nclass = 32          # Embedding (latent) dimension
#     dropout = 0.5
#     alpha = 0.2
#     nheads = 8
#     n_clusters = 3       # Underlying number of clusters (cancer subtypes)

#     # Generate common ground-truth cluster labels for all omics
#     labels = torch.randint(0, n_clusters, (n_nodes,))

#     # Generate synthetic omics data (mRNA, miRNA, DNA methylation)
#     mrna_data = generate_synthetic_omics(n_nodes, n_features, labels, noise=0.1)
#     mirna_data = generate_synthetic_omics(n_nodes, n_features, labels, noise=0.1)
#     dnam_data  = generate_synthetic_omics(n_nodes, n_features, labels, noise=0.1)

#     # Compute an adjacency (similarity) matrix for each omics and then average them.
#     sigma = 1.0
#     A_mrna = compute_adjacency_from_data(mrna_data, sigma)
#     A_mirna = compute_adjacency_from_data(mirna_data, sigma)
#     A_dnam = compute_adjacency_from_data(dnam_data, sigma)
#     A = (A_mrna + A_mirna + A_dnam) / 3.0

#     # Initialize the MultiGATAE model.
#     model = MultiGATAE(nfeat=n_features, nhid=nhid, nclass=nclass, dropout=dropout,
#                        alpha=alpha, nheads=nheads, npatient=n_nodes)
#     optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

#     # Training loop
#     model.train()
#     epochs = 200
#     for epoch in range(epochs):
#         optimizer.zero_grad()
#         adj_reconstructed, z = model(mrna_data, mirna_data, dnam_data, A)
#         loss = F.mse_loss(adj_reconstructed, A)
#         loss.backward()
#         optimizer.step()
#         if (epoch + 1) % 20 == 0:
#             print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

#     # After training, use K-means on the latent embedding z to get cluster assignments.
#     model.eval()
#     with torch.no_grad():
#         _, z = model(mrna_data, mirna_data, dnam_data, A)
#     z_np = z.detach().numpy()
#     kmeans = KMeans(n_clusters=n_clusters, random_state=0).fit(z_np)
#     pred_labels = kmeans.labels_
#     print("K-means Clustering Labels:", pred_labels)
#     print("Ground Truth Labels:      ", labels.numpy())

# if __name__ == "__main__":
#     main()


In [5]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt


def main():
    # Hyperparameters
    n_nodes = 100        # Number of patients/samples
    n_features = 500     # Feature dimension (for each omics)
    nhid = 64            # Hidden units per attention head
    nclass = 32          # Embedding (latent) dimension
    dropout = 0.5
    alpha = 0.2
    nheads = 8
    n_clusters = 3       # Underlying number of clusters (cancer subtypes)

    # Generate common ground-truth cluster labels for all omics
    labels = torch.randint(0, n_clusters, (n_nodes,))

    # Generate synthetic omics data (mRNA, miRNA, DNA methylation)
    mrna_data = generate_synthetic_omics(n_nodes, n_features, labels, noise=0.1)
    mirna_data = generate_synthetic_omics(n_nodes, n_features, labels, noise=0.1)
    dnam_data  = generate_synthetic_omics(n_nodes, n_features, labels, noise=0.1)

    # Compute an adjacency (similarity) matrix for each omics and then average them.
    sigma = 1.0
    A_mrna = compute_adjacency_from_data(mrna_data, sigma)
    A_mirna = compute_adjacency_from_data(mirna_data, sigma)
    A_dnam = compute_adjacency_from_data(dnam_data, sigma)
    A = (A_mrna + A_mirna + A_dnam) / 3.0

    # Initialize the MultiGATAE model.
    model = MultiGATAE(nfeat=n_features, nhid=nhid, nclass=nclass, dropout=dropout,
                       alpha=alpha, nheads=nheads, npatient=n_nodes)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

    # Training loop
    model.train()
    epochs = 60
    for epoch in range(epochs):
        optimizer.zero_grad()
        adj_reconstructed, z = model(mrna_data, mirna_data, dnam_data, A)
        loss = F.mse_loss(adj_reconstructed, A)
        loss.backward()
        optimizer.step()
        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

    # After training, use K-means on the latent embedding z to get cluster assignments.
    model.eval()
    with torch.no_grad():
        _, z = model(mrna_data, mirna_data, dnam_data, A)
    z_np = z.detach().numpy()
    kmeans = KMeans(n_clusters=n_clusters, random_state=0).fit(z_np)
    pred_labels = kmeans.labels_
    print("K-means Clustering Labels:", pred_labels)
    print("Ground Truth Labels:      ", labels.numpy())

    # Compute the silhouette score if there are at least 2 clusters
    if len(set(pred_labels)) > 1:
        silhouette_avg = silhouette_score(z_np, pred_labels)
        print(f"Silhouette Score: {silhouette_avg:.4f}")

        # Plot the silhouette score
        plt.figure()
        plt.bar(range(len(pred_labels)), silhouette_score(z_np, pred_labels, metric='euclidean'))
        plt.xlabel('Cluster')
        plt.ylabel('Silhouette Score')
        plt.title('Silhouette Score for K-means Clustering')
        plt.show()
    else:
        print("Silhouette score cannot be computed, only one cluster found.")

if __name__ == "__main__":
    main()

Epoch 20/60, Loss: 56317762863104.0000
Epoch 40/60, Loss: 29907665027072.0000
Epoch 60/60, Loss: 7257640665088.0000
K-means Clustering Labels: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Ground Truth Labels:       [0 0 0 0 0 1 1 1 2 0 2 0 0 0 1 1 0 0 0 1 0 2 1 2 0 0 1 2 0 1 0 1 1 1 0 0 1
 0 2 0 0 1 0 1 0 0 1 0 2 2 1 0 2 2 1 2 1 2 0 1 1 0 2 2 1 1 2 1 2 2 1 0 1 0
 0 0 0 1 0 1 1 2 0 0 2 0 0 0 2 0 0 2 2 1 0 2 2 0 0 1]
Silhouette score cannot be computed, only one cluster found.


c:\Users\praba\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
c:\Users\praba\anaconda3\Lib\site-packages\sklearn\base.py:1351: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


In [4]:
!pip install --upgrade mofapy2



[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   - -------------------------------------- 0.0/1.0 MB 495.5 kB/s eta 0:00:02
   -- ------------------------------------- 0.1/1.0 MB 660.6 kB/s eta 0:00:02
   ---- ----------------------------------- 0.1/1.0 MB 590.8 kB/s eta 0:00:02
   ---- ----------------------------------- 0.1/1.0 MB 602.4 kB/s eta 0:00:02
   ------ --------------------------------- 0.2/1.0 MB 573.4 kB/s eta 0:00:02
   ------- -------------------------------- 0.2/1.0 MB 588.1 kB/s eta 0:00:02
   -------- ------------------------------- 0.2/1.0 MB 599.0 kB/s eta 0:00:02
   --------- ------------------------------ 0.2/1.0 MB 600.7 kB/s eta 0:00:02
   ---------- ----------------------------- 0.3/1.0 MB 585.8 kB/s eta 0:00:02
   ------------ --------------------------- 0.3/1.0 MB 613.6 kB/s eta 0:00:02
   ------------ --------------------------- 0.3/1.0 MB 596.5 kB/s eta 0:00:02
   

In [6]:
devtools::install_github("bioFAM/MOFA2", build_opts = c("--no-resave-data --no-build-vignettes"))


SyntaxError: invalid syntax (1389427555.py, line 1)

In [8]:
!pip install scanpy



  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tf-nightly-intel 2.17.0.dev20240229 requires keras-nightly~=3.1.0.dev, which is not installed.
tables 3.8.0 requires blosc2~=2.0.0, which is not installed.
tables 3.8.0 requires cython>=0.29.21, which is not installed.
tf-nightly-intel 2.17.0.dev20240229 requires ml-dtypes~=0.3.1, but you have ml-dtypes 0.5.1 which is incompatible.
jax 0.5.0 requires numpy>=1.25, but you have numpy 1.24.4 which is incompatible.
jaxlib 0.5.0 requires numpy>=1.25, but you have numpy 1.24.4 which is incompatible.
langchain 0.0.225 requires pydantic<2,>=1, but you have pydantic 2.10.6 which is incompatible.
tensorflow-intel 2.15.0 requires ml-dtypes~=0.2.0, but you have ml-dtypes 0.5.1 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 2


   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
    --------------------------------------- 0.0/2.1 MB 186.2 kB/s eta 0:00:12
    --------------------------------------- 0.0/2.1 MB 186.2 kB/s eta 0:00:12
    --------------------------------------- 0.0/2.1 MB 140.3 kB/s eta 0:00:15
    --------------------------------------- 0.0/2.1 MB 140.3 kB/s eta 0:00:15
   - -------------------------------------- 0.1/2.1 MB 163.6 kB/s eta 0:00:13
   - -------------------------------------- 0.1/2.1 MB 163.6 kB/s eta 0:00:13
   - -------------------------------------- 0.1/2.1 MB 151.3 kB/s eta 0:00:14
   - -------------------------------------- 0.1/2.1 MB 151.3 kB/s eta 0:00:14
   - -------------------------

In [1]:
import numpy as np
import pandas as pd
import scanpy as sc

import matplotlib.pyplot as plt
import seaborn as sns



In [2]:
import muon as mu

AttributeError: module 'tensorflow._api.v2.compat.v2.__internal__' has no attribute 'register_load_context_function'